In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, LongType, StringType
import pandas as pd

1. dim_date

In [0]:
# 1. Recreate the business-day range (2000-today)
date_range = pd.date_range(start="2000-01-01", end=pd.Timestamp.today(), freq="B")
date_df = pd.DataFrame({"Full_Date": date_range})

# 2. Convert to Spark, cast explicitly, enrich with sorting numbers, and order columns
dim_date_spark = spark.createDataFrame(date_df)
dim_date_spark = (
    dim_date_spark
    .withColumn("Date_Key", F.date_format("Full_Date", "yyyyMMdd").cast(LongType()))
    .withColumn("Full_Date", F.col("Full_Date").cast(DateType()))
    .withColumn("Day_of_Week", F.date_format("Full_Date", "EEEE"))
    .withColumn("Day_Number_Of_Week", F.dayofweek("Full_Date")) # For custom weekday sorting
    .withColumn("Month_Name", F.date_format("Full_Date", "MMMM"))
    .withColumn("Month_Number", F.month("Full_Date")) # For chronological month sorting
    .withColumn("Quarter", F.concat(F.lit("Q"), F.quarter("Full_Date")))
    .withColumn("Year", F.year("Full_Date"))
)

# 3. Write to Gold layer
(
    dim_date_spark.write.format("delta")
    .mode("overwrite")
    .saveAsTable("gold.dim_date")
)

print("Successfully created and populated gold.dim_date!")

2. dim_region

In [0]:
# 1. Define region metadata
regions_data = [
    (1, "USA", "United States", "S&P 500", "USD"),
    (2, "EMU", "Euro Area", "STOXX Europe 50", "EUR"),
    (3, "CHN", "China", "Shanghai Composite", "CNY"),
    (4, "JPN", "Japan", "Nikkei 225", "JPY"),
    (5, "IND", "India", "Nifty 50", "INR"),
]

columns = [
    "Region_Key",
    "Country_Code",
    "Region_Name",
    "Stock_Exchange",
    "Currency_Code",
]

# 2. Create Spark DataFrame
dim_region_spark = spark.createDataFrame(regions_data, columns)

# 3. Write to Gold layer
(
    dim_region_spark.write.format("delta")
    .mode("overwrite")
    .saveAsTable("gold.dim_region")
)

print("Successfully created and populated gold.dim_region!")

3. fact_daily_stock_performance

In [0]:
# 1. Read individual cleaned Silver stock tables and assign their Region_Key
usa_df = spark.read.table("silver.sp500_cleaned").withColumn("Region_Key", F.lit(1))
eu_df = spark.read.table("silver.stoxx_eu_cleaned").withColumn("Region_Key", F.lit(2))
chn_df = spark.read.table("silver.ssec_cleaned").withColumn("Region_Key", F.lit(3))
jpn_df = spark.read.table("silver.nikkei_cleaned").withColumn("Region_Key", F.lit(4))
ind_df = spark.read.table("silver.nsei_cleaned").withColumn("Region_Key", F.lit(5))

# 2. Union all regional dataframes into a single master stock dataframe
union_stock_df = usa_df.unionByName(eu_df).unionByName(chn_df).unionByName(jpn_df).unionByName(ind_df)

# 3. Read dim_date to map the calendar Date to Date_Key
dim_date = spark.read.table("gold.dim_date")

# 4. Join stock data with dim_date, select required columns, and cast precision
fact_stock_df = (
    union_stock_df
    .join(dim_date, union_stock_df["Date"] == dim_date["Full_Date"], "inner")
    .select(
        F.monotonically_increasing_id().alias("Stock_Key"),
        "Date_Key",
        "Region_Key",
        F.col("Open").cast("decimal(18,2)").alias("Open"),
        F.col("Close").cast("decimal(18,2)").alias("Close"),
        F.col("High").cast("decimal(18,2)").alias("High"),
        F.col("Low").cast("decimal(18,2)").alias("Low"),
        F.col("Volume").cast("bigint").alias("Volume")
    )
)

# 5. Write to the Gold layer fact table
(
    fact_stock_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.fact_daily_stock_performance")
)

# Optional: Optimize and Z-order for high-performance querying
spark.sql("OPTIMIZE gold.fact_daily_stock_performance ZORDER BY (Date_Key, Region_Key)")

print("Successfully created and populated gold.fact_daily_stock_performance!")

4. fact_yearly_macro_indicators

In [0]:
# 1. Read the cleaned Silver macro table
macro_silver_df = spark.read.table("silver.gdp_macro_cleaned")

# 2. Unpivot the country columns into rows using a stack expression
unpivot_expr = "stack(5, 'USA', USA, 'EMU', EMU, 'CHN', CHN, 'JPN', JPN, 'IND', IND) as (Country_Code, GDP_Growth_Pct)"
macro_long_df = macro_silver_df.select("Year", F.expr(unpivot_expr))

# 3. Read dim_region to map Country_Code to Region_Key
dim_region = spark.read.table("gold.dim_region")
macro_with_region = macro_long_df.join(
    dim_region, macro_long_df["Country_Code"] == dim_region["Country_Code"], "inner"
)

# 4. Build yearly macro fact table at the true Year grain
fact_macro_df = (
    macro_with_region
    .select(
        F.monotonically_increasing_id().alias("Macro_Key"),
        F.col("Year").cast("integer").alias("Year"),
        "Region_Key",
        F.col("GDP_Growth_Pct").cast("decimal(18,2)").alias("GDP_Growth_Pct")
    )
)

# 5. Write to the Gold layer fact table with schema-overwrite protection
(
    fact_macro_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.fact_yearly_macro_indicators")
)

# Optional: Optimize and Z-order by Year and Region_Key for fast analytical queries
spark.sql(
    "OPTIMIZE gold.fact_yearly_macro_indicators ZORDER BY (Year, Region_Key)"
)

print("Successfully created and populated gold.fact_yearly_macro_indicators at the yearly grain!")